# Image Prefill — Sandbox v2 (T8 resolver)

Harness over [`sandbox/prefill_lib.py`](prefill_lib.py) with the T8 upgrades:
`item_kind` extraction + kind gating, species-facet-first add-on matching,
`first_year`/`last_year` era rejection, per-item add-on tiers (no `_min_tier`),
county-number ⇒ PA prior, serial O/0 correction, `context_text`, era suppression
when a year resolves, and the constrained **second pass** for unmatched add-ons
(`P.SECOND_PASS`, fires on misses only).

**Run order**
1. Seed the DB once: `python manage.py seed_states && python manage.py seed_geographic_units && python manage.py seed_license_types`
2. Put `ANTHROPIC_API_KEY` in the repo-root `.env`.
3. Run the cells top to bottom.

Edit `prefill_lib.py` and re-run — `autoreload` picks up changes without restarting the kernel.


In [ ]:
%pip install anthropic python-dotenv pillow rapidfuzz pandas matplotlib

In [ ]:
# Bootstrap: put repo root + sandbox on path, load the library (DB-backed)
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'manage.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (str(ROOT), str(ROOT / 'sandbox')):
    if p not in sys.path:
        sys.path.insert(0, p)

%load_ext autoreload
%autoreload 2
import prefill_lib as P

REF = P.ReferenceData()           # taxonomy snapshot from the Django ORM
CLIENT = P.get_client()           # None if ANTHROPIC_API_KEY is missing
addons = [c for c in REF.lt.values()]
print('PA counties:', len(REF.geo_by_state.get('PA', [])),
      '| license types:', sum(len(v) for v in REF.lt.values()),
      '| addon rows with year floors:', sum(1 for k, v in REF.lt.items() if k[1] == 'addon_type' for c in v if c.get('first_year')),
      '| client:', 'ready' if CLIENT else 'NO API KEY')
print('Knobs -> SECOND_PASS:', P.SECOND_PASS, '| ALLOW_INFERENCE:', P.ALLOW_INFERENCE,
      '| FUZZY_FLOOR:', P.FUZZY_FLOOR, '| MAX_IMAGE_EDGE:', P.MAX_IMAGE_EDGE)


## Run extraction + resolution on the test images

In [ ]:
import base64, glob, json
from io import BytesIO
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

SCALAR_FIELDS = ['item_kind', 'state', 'license_year', 'era_guess', 'geographic_unit',
                 'is_statewide', 'residency', 'holder_eligibility', 'activity_scope',
                 'duration', 'material', 'shape', 'colors', 'serial_number']
TIER_BG = {'high': '#d9f2d9', 'medium': '#fff3cd', 'low': '#ffe5cc', 'unmatched': '#f8d7da'}


def _row(field, d):
    via = 'inferred' if d.get('inferred') else ''
    if d.get('second_pass'):
        via = (via + ' 2nd-pass').strip()
    resolved = str(d.get('name') or '')
    if d.get('corrected_from'):
        resolved += ' (corrected from ' + d['corrected_from'] + ')'
    return {'field': field, 'source_text': str(d.get('source_text') or ''),
            'resolved': resolved, 'tier': d.get('tier'),
            'conf': round(d.get('conf', 0), 2), 'via': via}


def render(sf, path, res, ex):
    rows = []
    for k in SCALAR_FIELDS:
        d = res['fields'].get(k) or {}
        if not d.get('source_text') and not d.get('value'):
            continue
        rows.append(_row(k, d))
    # per-item add-ons are the contract — one row each, misses route to suggestions
    for i, item in enumerate(res['fields'].get('addon_type', {}).get('items', []), 1):
        rows.append(_row('addon[' + str(i) + ']', item))
    styled = (pd.DataFrame(rows).style.hide(axis='index')
              .map(lambda v: 'background-color:' + TIER_BG.get(v, '#fff'), subset=['tier']))
    with Image.open(path) as im:
        thumb = im.copy()
    thumb.thumbnail((300, 300))
    buf = BytesIO()
    thumb.convert('RGB').save(buf, 'JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    total_cost = ex['cost_usd'] + res.get('second_pass_cost_usd', 0.0)
    meta = format(total_cost * 1000, '.2f') + ' milli-$ / ' + format(ex['latency_ms'], '.0f') + ' ms'
    if res.get('second_pass_cost_usd'):
        meta += ' (incl. 2nd pass)'
    banner = ('<div style="background:#e7f1ff;border:1px solid #9ec5fe;border-radius:6px;'
              'padding:4px 10px;margin:4px 0">Looks like <b>multiple items</b> — suggest a lot listing (10.15)</div>'
              if res.get('lot_detected') else '')
    img = '<img src="data:image/jpeg;base64,' + b64 + '" style="max-width:300px;border:1px solid #ccc;border-radius:6px">'
    head = '<b>' + sf + ': ' + path.name + '</b><br><span style="color:#666">' + meta + '</span><br>'
    display(HTML('<div style="display:flex;gap:20px;align-items:flex-start"><div>'
                 + head + banner + img + '</div><div>' + styled.to_html() + '</div></div>'))
    details = '<details><summary>raw transcription</summary><pre style="white-space:pre-wrap">' \
              + (res['raw_text'] or '(empty)') + '</pre></details>'
    if res.get('context_text'):
        details += '<details><summary>context text (mat/annotations — weak evidence only)</summary>' \
                   '<pre style="white-space:pre-wrap">' + res['context_text'] + '</pre></details>'
    display(HTML(details))


def run_all(limit=None, use_cache=False):
    cache_path = ROOT / 'sandbox' / 't8_raws.json'
    cache = json.loads(cache_path.read_text()) if (use_cache and cache_path.exists()) else {}
    dirs = [('listing', ROOT / 'media' / 'listings'), ('collection', ROOT / 'media' / 'collections')]
    imgs = [(sf, Path(p)) for sf, d in dirs for p in sorted(glob.glob(str(d / '*')))
            if Path(p).suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
    if limit:
        imgs = imgs[:limit]
    out, cost, raws = [], 0.0, {}
    for i, (sf, path) in enumerate(imgs, 1):
        print('[' + str(i) + '/' + str(len(imgs)) + '] ' + sf + ': ' + path.name)
        if path.name in cache:
            ex = cache[path.name]
        else:
            ex = P.extract(str(path), CLIENT)
        if 'error' in ex:
            print('  error:', ex['error'])
            continue
        raws[path.name] = {'raw': ex['raw'], 'cost_usd': ex['cost_usd'], 'latency_ms': ex['latency_ms']}
        res = P.resolve(ex['raw'], REF, CLIENT)
        cost += ex['cost_usd'] + res.get('second_pass_cost_usd', 0.0)
        out.append({'sf': sf, 'name': path.name, 'ex': ex, 'res': res})
        render(sf, path, res, ex)
    cache_path.write_text(json.dumps(raws, indent=1))
    avg = cost / max(len(out), 1) * 1000
    print('Done. ' + str(len(out)) + ' images, total $' + format(cost, '.4f')
          + ', avg ' + format(avg, '.2f') + ' milli-$/image. Raw extractions cached to sandbox/t8_raws.json'
          + ' — re-run with run_all(use_cache=True) to iterate on the resolver for free.')
    return out


results = run_all()


## Gold-set scoring (R6)

Label the images into `sandbox/gold.json` (schema per the research doc — one object per
image filename) and this cell reports **per-field precision per tier**, which is what turns
resolver changes into before/after numbers. Until the file exists this cell just tells you so.


In [ ]:
from collections import defaultdict

GOLD = ROOT / 'sandbox' / 'gold.json'
if not GOLD.exists():
    print('sandbox/gold.json not found - label the images first (R6). Example entry:')
    print(json.dumps({'test_listing_image_001.png': {
        'state': 'PA', 'license_year': 1923, 'geographic_unit': 'Lancaster',
        'item_kind': 'license', 'serial_number': '53101H',
        'addons': ['Turkey Tag'], 'material': 'Metal Tag', 'shape': 'Rectangle',
        'colors': ['forest_green'], 'residency': 'Resident'}}, indent=1))
else:
    gold = json.loads(GOLD.read_text())
    SCORE_FIELDS = ['item_kind', 'state', 'license_year', 'geographic_unit', 'residency',
                    'holder_eligibility', 'activity_scope', 'duration', 'material',
                    'shape', 'serial_number']
    stats = defaultdict(lambda: {'n': 0, 'ok': 0})   # (field, tier) -> counts

    def norm(v):
        return str(v or '').strip().lower()

    for r in results:
        g = gold.get(r['name'])
        if not g:
            continue
        f = r['res']['fields']
        for k in SCORE_FIELDS:
            d = f.get(k) or {}
            if d.get('value') is None or k not in g:
                continue
            got = norm(d.get('name') if k != 'license_year' else d.get('value'))
            want = norm(g[k])
            s = stats[(k, d['tier'])]
            s['n'] += 1
            s['ok'] += int(got == want or (want and want in got) or (got and got in want))
        want_addons = {norm(a) for a in g.get('addons', [])}
        for item in (f.get('addon_type') or {}).get('items', []):
            if item.get('value') is None:
                continue
            s = stats[('addons', item['tier'])]
            s['n'] += 1
            s['ok'] += int(norm(item['name']) in want_addons)

    rows = [{'field': k, 'tier': t, 'n': v['n'], 'correct': v['ok'],
             'precision': round(v['ok'] / v['n'], 3)}
            for (k, t), v in sorted(stats.items()) if v['n']]
    df = pd.DataFrame(rows)
    display(df)
    by_tier = df.groupby('tier').agg(n=('n', 'sum'), correct=('correct', 'sum'))
    by_tier['precision'] = (by_tier['correct'] / by_tier['n']).round(3)
    print('\nPer-tier overall (target: high >= 0.98):')
    display(by_tier)


## Scratchpad

Single image, or iterate on the resolver for free against the cached raws:

```python
ex = P.extract('media/collections/test_collection_image_005.png', CLIENT)
res = P.resolve(ex['raw'], REF, CLIENT)
ex['raw'], res['fields']['addon_type']['items']

# resolver-only iteration (no API cost):
results = run_all(use_cache=True)
```
